# Jasper Lim
# 9117038

## CSCN8000 - Assignment 2

### Boston Housing MEDV Prediction

## Setup and configuration

Place `housing.csv` beside this notebook. It uses NumPy, pandas, scikit-learn, and XGBoost; install any missing packages in the active notebook kernel before running the cells. The data-loading cell checks for the input file and loads the housing feature names and MEDV target.

In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LinearRegression, LogisticRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor

columns = [
    "CRIM", "ZN", "INDUS", "CHAS", "NOX", "RM", "AGE", "DIS",
    "RAD", "TAX", "PTRATIO", "B", "LSTAT", "MEDV",
]
data_path = Path("housing.csv")
if not data_path.exists():
    raise FileNotFoundError("Place housing.csv in the same folder as this notebook.")

housing = pd.read_csv(data_path, sep=r"\s+", header=None, names=columns)
features = columns[:-1]
X, y = housing[features], housing["MEDV"]


## Question 1 - Data preprocessing

Check missing values and feature outliers using the IQR rule. Outliers are reported and retained. The model pipelines median-impute and standardize features using training data only.

## Question 2 - Training and test split

Use 80% of the samples for training and the remaining 20% for testing. The next cell performs the split and reports the sample counts.

In [4]:
print(f"Data shape: {housing.shape}; duplicate rows: {housing.duplicated().sum()}")
print("Missing values by column:")
display(housing.isna().sum().to_frame("missing"))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.80, random_state=42
)
print(f"Training samples: {len(X_train)} ({len(X_train) / len(X):.1%}); test samples: {len(X_test)} ({len(X_test) / len(X):.1%})")

q1, q3 = X_train.quantile(0.25), X_train.quantile(0.75)
iqr = q3 - q1
outliers = ((X_train < q1 - 1.5 * iqr) | (X_train > q3 + 1.5 * iqr)).sum()
print("Training feature outliers by IQR (retained):")
display(outliers[outliers > 0].to_frame("outlier_count"))



Data shape: (506, 14); duplicate rows: 0
Missing values by column:


,missing
CRIM,0
ZN,0
INDUS,0
CHAS,0
NOX,0
RM,0
AGE,0
DIS,0
RAD,0
TAX,0


Training samples: 404 (79.8%); test samples: 102 (20.2%)
Training feature outliers by IQR (retained):


,outlier_count
CRIM,61
ZN,36
CHAS,29
RM,24
DIS,3
B,63
LSTAT,7


## Question 3 - Model training

Train multilinear regression, XGBoost, decision tree, random forest, logistic regression, Lasso (L1), and Ridge (L2). Logistic regression is trained on MEDV quantile classes; its class probabilities are converted into continuous MEDV estimates.

In [5]:
def regression_pipeline(estimator):
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), estimator)

class LogisticMEDVRegressor:
    """Use logistic quantile classification probabilities for continuous MEDV estimates."""

    def __init__(self):
        self.pipeline = make_pipeline(
            SimpleImputer(strategy="median"),
            StandardScaler(),
            LogisticRegression(max_iter=3000, random_state=42),
        )

    def fit(self, X_fit, y_fit):
        y_values = pd.Series(y_fit, index=X_fit.index)
        bins = pd.qcut(y_values, q=10, labels=False, duplicates="drop")
        self.bin_values_ = y_values.groupby(bins).mean()
        self.pipeline.fit(X_fit, bins)
        classes = self.pipeline[-1].classes_
        self.class_values_ = np.array([self.bin_values_.loc[label] for label in classes])
        return self

    def predict(self, X_predict):
        return self.pipeline.predict_proba(X_predict) @ self.class_values_

models = {
    "Linear Regression": regression_pipeline(LinearRegression()),
    "XGBoost": regression_pipeline(XGBRegressor(
        objective="reg:squarederror", n_estimators=300, max_depth=3,
        learning_rate=0.05, subsample=0.9, colsample_bytree=0.9,
        random_state=42, n_jobs=-1,
    )),
    "Decision Tree": regression_pipeline(DecisionTreeRegressor(random_state=42)),
    "Random Forest": regression_pipeline(
        RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)
    ),
    "Logistic Regression": LogisticMEDVRegressor(),
    "Lasso (L1)": regression_pipeline(Lasso(alpha=0.1, max_iter=10000)),
    "Ridge (L2)": regression_pipeline(Ridge(alpha=1.0)),
}
for model in models.values():
    model.fit(X_train, y_train)


## Question 4 - Test-set predictions

Generate a MEDV prediction for every test sample with each trained model.

In [6]:
predictions = {name: model.predict(X_test) for name, model in models.items()}



## Question 5 - Predictive system

Use the fitted models to predict MEDV for a new sample. The example uses a held-out test row, and `predict_medv` can also be called with a feature dictionary.

In [7]:
def predict_medv(sample):
    """Return MEDV estimates from each fitted model for one feature dictionary."""
    missing_features = set(features) - set(sample)
    if missing_features:
        raise ValueError(f"Missing required features: {sorted(missing_features)}")
    sample_frame = pd.DataFrame([sample], columns=features)
    return pd.Series(
        {name: float(model.predict(sample_frame)[0]) for name, model in models.items()},
        name="Predicted MEDV ($1,000s)",
    )

new_sample = X_test.iloc[0].to_dict()  # Held-out example; not used for training.
print("Example new-sample predictions:")
display(predict_medv(new_sample).to_frame())

Example new-sample predictions:


,"Predicted MEDV ($1,000s)"
Linear Regression,28.996724
XGBoost,23.865122
Decision Tree,26.400000
Random Forest,22.926667
Logistic Regression,26.965334
Lasso (L1),28.618484
Ridge (L2),28.973076


## Question 6 - Evaluation and model comparison

Compare within-$5,000 accuracy, Mean Absolute Error, R-squared, and Root Mean Squared Error. Since MEDV is continuous, accuracy is defined here as the percentage of predictions within $5,000 of the actual value. The table is sorted by that accuracy measure.

In [8]:
tolerance = 5.0  # MEDV units are $1,000s; tolerance is $5,000.
metrics = pd.DataFrame({
    name: {
        "Accuracy within $5k (%)": 100 * np.mean(np.abs(y_test.to_numpy() - pred) <= tolerance),
        "MAE ($1,000s)": mean_absolute_error(y_test, pred),
        "R-squared": r2_score(y_test, pred),
        "RMSE ($1,000s)": np.sqrt(mean_squared_error(y_test, pred)),
    }
    for name, pred in predictions.items()
}).T.sort_values("Accuracy within $5k (%)", ascending=False)
print("Test-set evaluation (best accuracy first):")
display(metrics.round(3))
print(f"Best by within-$5k accuracy: {metrics.index[0]}")



Test-set evaluation (best accuracy first):


,Accuracy within $5k (%),"MAE ($1,000s)",R-squared,"RMSE ($1,000s)"
Random Forest,95.098,2.082,0.877,3.005
XGBoost,94.118,1.899,0.896,2.766
Logistic Regression,90.196,2.626,0.730,4.453
Lasso (L1),86.275,3.242,0.650,5.065
Linear Regression,86.275,3.189,0.669,4.929
Ridge (L2),86.275,3.186,0.668,4.931
Decision Tree,83.333,2.725,0.757,4.221


Best by within-$5k accuracy: Random Forest


## Question 7 - Save predictions

Write actual MEDV and every model's test-set predictions to a single CSV file named `housing_test_predictions.csv` in the notebook folder.

In [9]:
test_predictions = pd.DataFrame({
    "row_index": X_test.index,
    "actual_MEDV": y_test,
    **{f"predicted_{name}": pred for name, pred in predictions.items()},
})
output_path = Path("housing_test_predictions.csv")
test_predictions.to_csv(output_path, index=False)
print(f"Saved {len(test_predictions)} test predictions to {output_path.resolve()}")



Saved 102 test predictions to C:\Users\User\Documents\conestoga\CSCN8000\CSCN8000-lab-submissions\students-submissions\9117038\a2\housing_test_predictions.csv


## Findings Summary

- The dataset contains 506 rows and 13 input features; no missing values or duplicate rows were found.
- The 80/20 split produced 404 training samples and 102 test samples. IQR outliers were identified in several features and retained.
- Random Forest had the highest within-$5,000 accuracy (95.098%). XGBoost had the lowest MAE (1.899), lowest RMSE (2.766), and highest R-squared (0.896).
- Metrics are reported on the held-out test set; error values are in MEDV units ($1,000s).

| Model | Accuracy within $5k (%) | MAE | R-squared | RMSE |
|---|---:|---:|---:|---:|
| Random Forest | 95.098 | 2.082 | 0.877 | 3.005 |
| XGBoost | 94.118 | 1.899 | 0.896 | 2.766 |
| Logistic Regression | 90.196 | 2.626 | 0.730 | 4.453 |
| Lasso (L1) | 86.275 | 3.242 | 0.650 | 5.065 |
| Linear Regression | 86.275 | 3.189 | 0.669 | 4.929 |
| Ridge (L2) | 86.275 | 3.186 | 0.668 | 4.931 |
| Decision Tree | 83.333 | 2.725 | 0.757 | 4.221 |